# 18 — PDF and Multimodal RAG

RAG over PDF documents and multimodal vision prompting.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage
from langchain.text_splitter import RecursiveCharacterTextSplitter

## Part 1: PDF RAG (Simulated)

In [ ]:
PAGES = [
    Document(page_content="Annual Report 2024 — Acme Corp\nExecutive Summary\nAcme Corp achieved record revenue of $4.2 billion in 2024, a 15% YoY increase. Our AI division grew by 45%.", metadata={"source": "annual_report.pdf", "page": 1}),
    Document(page_content="Financial Highlights\nRevenue: $4.2B (up 15% YoY)\nNet Income: $890M (up 22%)\nR&D Spending: $1.1B (26% of revenue)\nEmployee Count: 12,500", metadata={"source": "annual_report.pdf", "page": 2}),
    Document(page_content="Product Updates\n1. Acme AI Platform v3.0 with multi-modal capabilities\n2. Cloud infrastructure expanded to 8 new regions\n3. Enterprise customer base grew to 2,400 companies", metadata={"source": "annual_report.pdf", "page": 3}),
]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50).split_documents(PAGES)
vectorstore = Chroma.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

def format_docs(docs):
    return "\n\n".join(f"[Page {d.metadata.get('page', '?')}] {d.page_content}" for d in docs)

for q in ["What was Acme Corp's revenue in 2024?", "How much did the company spend on R&D?"]:
    retrieved = retriever.invoke(q)
    answer = (ChatPromptTemplate.from_template("Answer from PDF context:\n{context}\n\nQuestion: {question}") | llm | StrOutputParser()).invoke({"context": format_docs(retrieved), "question": q})
    pages = [str(d.metadata.get("page", "?")) for d in retrieved]
    print(f"Q: {q}\nA: {answer}\nSources: pages {', '.join(pages)}\n")

vectorstore.delete_collection()

## Part 2: Multimodal Vision

In [ ]:
message = HumanMessage(content=[
    {"type": "text", "text": "Describe this image in one sentence."},
    {"type": "image_url", "image_url": {"url": "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c3/Python-logo-notext.svg/200px-Python-logo-notext.svg.png"}},
])

response = llm.invoke([message])
print(f"Vision response: {response.content}")